In [10]:
import pandas as pd 
import matplotlib as mlt
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import nbformat



In [11]:
all_df = pd.read_csv('credit_fraud.csv')
 
all_df.dtypes
# all_df
all_df['age_numeric'] = pd.to_numeric(all_df['age'], errors='coerce')

fraud_df = all_df[(all_df['is_fraud'] == 1) & (all_df['age_numeric'] > 18)]
no_fraud_df = all_df[(all_df['is_fraud'] == 0) & (all_df['age_numeric'] > 18)]

all_df
fraud_df
all_df['age'].value_counts(dropna=False)

fraud_df.dtypes

age                           str
transaction_amount        float64
account_balance           float64
num_transactions_today    float64
is_foreign_transaction    float64
transaction_hour          float64
prev_fraud_flag           float64
merchant_distance_km      float64
merchant_risk_score       float64
is_fraud                    int64
age_numeric               float64
dtype: object

In [21]:
all_df['transaction_hour']

0         9.0
1         4.0
2         0.0
3        17.0
4         NaN
         ... 
32295     6.0
32296    18.0
32297     9.0
32298     0.0
32299    11.0
Name: transaction_hour, Length: 32300, dtype: float64

In [12]:
fig = px.scatter_3d(all_df, x="is_fraud", y="age_numeric", z="account_balance",
                    hover_name=all_df.index,
                    opacity=0.5)
fig.show()

In [13]:
corr = all_df[['is_fraud', 'merchant_risk_score']].corr()
corr

,is_fraud,merchant_risk_score
is_fraud,1.000000,0.218451
merchant_risk_score,0.218451,1.000000


The idea here is to bin transaction_amount into ranges and then create two new columns:

amount_bracket — which bucket the transaction falls into

amount_bracket_fraud_rate — the fraud rate for that bucket (useful as a numeric feature for ML)

In [14]:
age_bins = [17, 35, 60, 75]
age_labels = ['Young Adult', 'Middle Aged', 'Senior']

In [22]:
bins = [0,2500, 5000, 7500, 10000, 15000]
labels = ['$0-2.5k', '$2.5-5k', '$5-7.5k', '$7.5-10k', '$10-15k']

age_bins = [17, 35, 60, 75]
age_labels = ['Young Adult', 'Middle Aged', 'Senior']

#BINNING DEBT BY FRAUD 
all_df["amount_bracket"] = pd.cut(all_df["transaction_amount"], bins=bins, labels=labels)
fraud_rate_map = all_df.groupby('amount_bracket', observed=True)['is_fraud'].mean().round(4)
all_df['amount_bracket_fraud_rate'] = all_df['amount_bracket'].map(fraud_rate_map)

#BINNING AGE BY COL FRAUD 
all_df["age_bracket"] = pd.cut(all_df['age_numeric'], bins= age_bins, labels=age_labels)
fraud_age_map = all_df.groupby('age_bracket', observed=True )['is_fraud'].mean().round(4)
all_df['age_amount_bracket_fraud'] = all_df['age_bracket'].map(fraud_age_map)

#CONVERTING THE MESSY DATA INTO A NUMERIC VALUE
all_df['amount_bracket_fraud_rate'] = pd.to_numeric(all_df['amount_bracket_fraud_rate'], errors='coerce')
all_df['age_amount_bracket_fraud'] = pd.to_numeric(all_df['age_amount_bracket_fraud'], errors='coerce')

#COMBINING THE TWO BINS INTO A NEW COLUMN
all_df["Total_Risk_Score"] = all_df['amount_bracket_fraud_rate'] + all_df['age_amount_bracket_fraud']

#USER INPUT 
user_age = int(input("Please enter your age: "))
user_balance = int(input("Please enter your account balance: "))

#COMPARING THE THE BINS CREATED EARLIER
u_amt_bracket = pd.cut([user_balance], bins=bins, labels=labels)[0]
u_age_bracket = pd.cut([user_age], bins=age_bins, labels=age_labels)[0]

#DETERMINING THE RISK PROFILE FOR THE USER
if pd.notna(u_amt_bracket) and pd.notna(u_age_bracket):
    risk_amt = fraud_rate_map[u_amt_bracket]
    risk_age = fraud_age_map[u_age_bracket]

    total_risk = (risk_amt + risk_age).round(4)

    print(f"\n--- Risk Profile ---")
    print(f"Age Group: {u_age_bracket}")
    print(f"Balance Group: {u_amt_bracket}")
    print(f"Combined Risk Score: {total_risk:.4%}")
    fig = px.scatter_3d(
            all_df, 
            x='age_numeric', 
            y='transaction_amount', 
            z='Total_Risk_Score',
            color='Total_Risk_Score',
            title="Fraud Risk Landscape",
            labels={'age_numeric': 'Age', 'transaction_amount': 'Balance', 'Total_Risk_Score': 'Risk Score'},
            opacity=0.6)
    fig.show() 
else:
    print("\nError: Age or Balance falls outside of defined brackets.")




--- Risk Profile ---
Age Group: Middle Aged
Balance Group: $7.5-10k
Combined Risk Score: 98.8600%


In [16]:
del corr
corr = all_df[['Total_Risk_Score', 'merchant_risk_score']].corr()
corr

,Total_Risk_Score,merchant_risk_score
Total_Risk_Score,1.000000,0.003978
merchant_risk_score,0.003978,1.000000
